# Q1-2026 Atrium SPAN cross-margin backtest

Reproducible notebook for the headline backtest claim. Inputs come from the
free Hyperliquid info API. Outputs are written to `q1-2026-backtest.json`
and pinned to IPFS. The IPFS hash + computed numbers are committed on-chain
via `ResearchAttestation.publish(...)`.

**Honesty rule:** every number on the Verifier landing page comes from this
notebook output, never from memory. If the data fetch fails or returns too
few rows, the run aborts rather than fabricate a result.

## Reproduce

```bash
pip install -e services/archive
python services/archive/src/span_backtest.py \
    --start 2026-01-01 --end 2026-03-31 \
    --output q1-2026-backtest.json
```

Then publish via Praetor CLI:

```bash
praetor backtest publish \
    --notebook services/archive/notebooks/q1-2026-backtest.ipynb \
    --ipfs-cid $(ipfs add q1-2026-backtest.json)
```

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from atrium_archive.span_backtest import (
    fetch_hyperliquid_trades, group_into_portfolios,
    isolated_margin_required, atrium_span_margin_required,
)

In [ ]:
trades = list(fetch_hyperliquid_trades(
    datetime(2026, 1, 1), datetime(2026, 3, 31), limit=2000
))
len(trades)

In [ ]:
portfolios = group_into_portfolios(trades)
for p in portfolios:
    p.isolated_margin_required = isolated_margin_required(p.positions)
    p.atrium_span_margin_required = atrium_span_margin_required(p.positions)

df = pd.DataFrame([{
    'trader': p.trader,
    'positions': len(p.positions),
    'isolated': p.isolated_margin_required,
    'atrium': p.atrium_span_margin_required,
    'saving_bps': p.saving_bps,
} for p in portfolios])
df.describe()

In [ ]:
df['saving_bps'].hist(bins=50)
plt.title('Q1-2026 SPAN collateral savings (bps)')
plt.xlabel('saving (bps)')
plt.ylabel('# portfolios')
plt.show()

print(f"Median saving: {df['saving_bps'].median():.0f} bps")
print(f"Mean saving:   {df['saving_bps'].mean():.0f} bps")
print(f"p10..p90:      {df['saving_bps'].quantile(0.1):.0f} bps .. {df['saving_bps'].quantile(0.9):.0f} bps")

## On-chain commitment

Once the notebook output is reviewed and signed off, commit the IPFS hash
via `ResearchAttestation.publish(...)`. The Verifier landing page then reads
the on-chain hash and renders the saving % verbatim.

Do not edit the rendered number on the landing page directly. The only
source of truth is this notebook output committed on chain.